# ① 전처리 — 문서 파싱 · 청크 분할
담당: 전처리 · 코드: `rag/preprocessing/parsing.py`, `rag/preprocessing/chunking.py`

`config.yaml` 의 `paths.input`(ZIP 또는 폴더)에 있는 PDF · HWP · HWPX 를 읽어 본문 · 표 · (선택) 그림 설명을 뽑고, 청크로 나눕니다.

| 저장 파일 | 읽는 노트북 |
|---|---|
| `artifacts/chunks.jsonl` (청크) | 02 검색, 03 검증 |
| `artifacts/docs.jsonl` (문서 목록 — 발주기관 라우팅용) | 02 검색 |
| `artifacts/parsed/` (문서별 파싱 결과 — 끊겨도 이어서 실행) | 01 전처리 (다시 실행할 때) |
| `artifacts/chunked/` (문서별 청크 — 끊겨도 이어서 실행) | 01 전처리 (다시 실행할 때) |

In [1]:
from rag.config import ARTIFACTS, ROOT, load_config
from rag.preprocessing.chunking import chunk_documents, chunk_summary, load_package, save_chunks
from rag.preprocessing.parsing import collect_inputs, parse_documents, parsing_summary
from rag.usage import usage_summary

cfg = load_config()

## 설정
초기값은 `config.yaml`에서 읽습니다. 아래 변수의 오른쪽을 원하는 값으로 바꾸고 설정 셀부터 다시 실행하세요.
`OCR = True`이면 선택한 비전 모델로 그림을 텍스트로 바꿉니다(OpenAI API, 유료). 기본값은 `False`이며, 이때 그림을 API로 보내지 않습니다.
청킹 크기는 본문 글자 수 기준이며, 변경 후 01부터 후속 노트북을 다시 실행해야 합니다.
`PREPROCESSED`는 기본값으로 증강 Golden dev 396 패키지의 검색 코퍼스(`data/Golden_Dev396_v12_Team_Package/corpus` — v12 98문서 · 18,989청크)를 그대로 읽습니다.
`None`이면 현재 파서를 사용하고, v12 · v11 전처리 패키지 폴더(`"data/RFP_v12"` 등)를 주면 그 패키지를 02 형식으로 옮깁니다.
패키지 모드에서는 저장된 OCR 텍스트를 그대로 읽으며 파싱·OCR·청크 재분할을 건너뜁니다. 패키지의 요구사항 소속 · 표 연결 정보도 함께 넘어가 02 검색이 씁니다.


In [2]:
PREPROCESSED = "data/Golden_Dev396_v12_Team_Package/corpus"  # None = 파서 사용, "data/RFP_v12" = v12 전처리 패키지
OCR = False
OCR_MODEL = cfg["ocr"]["model"]                    # 예: "gpt-5-mini"
OCR_REASONING_EFFORT = cfg["ocr"]["reasoning_effort"]  # 예: "low"
OCR_MAX_TOKENS = cfg["ocr"]["max_tokens"]          # 출력·추론 토큰 한도
OCR_PRICES = cfg["ocr"]["usd_per_1m_tokens"]     # 입력 · 캐시 입력 · 출력 단가
CHUNK_SIZE = cfg["chunking"]["chunk_size"]         # 예: 700

cfg["ocr"].update(model=OCR_MODEL, reasoning_effort=OCR_REASONING_EFFORT, max_tokens=OCR_MAX_TOKENS,
                  usd_per_1m_tokens=OCR_PRICES)
cfg["chunking"]["chunk_size"] = CHUNK_SIZE
cfg["ocr"], cfg["chunking"]


({'model': 'gpt-5-mini',
  'max_tokens': 2048,
  'reasoning_effort': 'low',
  'usd_per_1m_tokens': [0.25, 0.025, 2.0]},
 {'chunk_size': 700,
  'overlap_ratio': 0.2,
  'table_size_factor': 2.0,
  'min_chunk_chars': 30})

## 1. 입력 파일 모으기
ZIP(안의 ZIP 포함)을 `artifacts/raw/` 에 풀고 PDF · HWP · HWPX 목록을 만듭니다.

In [3]:
if PREPROCESSED is None:
    files = collect_inputs(ROOT / cfg["paths"]["input"], ARTIFACTS / "raw")
    print(f"입력 파일: {len(files)}")

## 2. 파싱
문서마다 쪽 · 표 · 그림 수와 표지 감지 결과를 봅니다.
문서 하나를 끝낼 때마다 결과를 `artifacts/parsed/` 에 저장합니다. OCR을 켜든 끄든 중간에 끊겨도 이 셀부터 다시 실행하면 끝낸 문서(`저장된 결과`)는 건너뛰고 이어서 합니다 — OCR 비용도 다시 들지 않습니다.
원본 파일이나 `parsing` · `ocr` 설정을 바꾸면 그 문서는 자동으로 다시 파싱합니다. 처음부터 다시 하려면 `artifacts/parsed/` 를 지우세요.

In [4]:
if PREPROCESSED is None:
    docs = parse_documents(files, cfg["parsing"], ocr=OCR, ocr_cfg=cfg["ocr"], cache_dir=ARTIFACTS / "parsed")
    display(parsing_summary(docs))
else:
    docs, chunks = load_package(ROOT / PREPROCESSED)
    print(f"패키지 문서: {len(docs)}, 청크: {len(chunks)}")

패키지 문서: 98, 청크: 18989


## 3. 청크 분할 · 저장
문서 하나를 나눌 때마다 결과를 `artifacts/chunked/` 에 저장합니다. 중간에 끊겨도 이 셀부터 다시 실행하면 끝낸 문서(`저장된 결과`)는 건너뛰고 이어서 나눕니다.
파싱 결과나 `CHUNK_SIZE` 등 `chunking` 설정이 바뀐 문서는 자동으로 다시 나눕니다. 처음부터 다시 하려면 `artifacts/chunked/` 를 지우세요.
`chunks.jsonl` · `docs.jsonl` 은 임시 파일에 쓴 뒤 바꿔치기하므로, 저장 도중 꺼져도 반쯤 쓰인 파일이 남지 않습니다.

In [5]:
if PREPROCESSED is None:
    chunks = chunk_documents(docs, cfg["chunking"], cache_dir=ARTIFACTS / "chunked")
save_chunks(docs, chunks, ARTIFACTS)
chunk_summary(chunks)

,청크 수,prose,table,requirement_summary,requirement_detail,image_text,image_ocr,평균 길이,중앙값,문장 중간 절단(%),섹션 부착(%)
0,18989,4930,8625,309,5101,6,18,397.5,279,70.6,63.3


## API 사용량
호출마다 `artifacts/api_usage.csv`에 자동 추가됩니다. 아래 표는 누적 사용량이며 OCR을 끄면 새 호출이 없습니다.

In [6]:
usage_summary("ocr")

,stage,evaluation,model,reasoning_effort,calls,input_tokens,cached_input_tokens,output_tokens,reasoning_tokens,cost_usd
